In [54]:
from ase.io.trajectory import Trajectory
from flonacomldft.collective_variables import get_collective_variables
import numpy as np

In [2]:
# path_database = '/mnt/home/amolina/ceph/project-database/andersen/'
# md_files = ['trajectories/27037382_is0_andersen.traj',  'trajectories/27037385_is1_andersen.traj']
# 
# trajs = {i: Trajectory(path_database + md_files[i]) for i in range(len(md_files))}

In [3]:
#temperature = {i: np.array([molecule.get_temperature() for molecule in trajs[i]]) for i in trajs.keys()}

In [4]:
#u = {i: np.array([molecule.get_potential_energy() for molecule in trajs[i]]) for i in trajs.keys()}

In [5]:
#cv = {i: np.stack([get_collective_variables(molecule) for molecule in trajs[i][:5]]) for i in trajs.keys()}

In [55]:
from flonacomldft.utils.io_utils import get_path, load_pickle_file
import torch

In [56]:
flow_models = [ load_pickle_file('dict_flow_model_is'+str(i)+'.pkl', get_path() + '/andersen/models/')['model'] for i in range(2)]
mlp_model = [load_pickle_file('dict_mlp_model_is'+str(i)+'.pkl', get_path() + '/andersen/models/')['model'] for i in range(2)]

In [57]:
from flonacomldft.models.mixture import Mixture

flow_model = Mixture(flow_models, torch.tensor([0.5, 0.5]))

In [58]:
import warnings
warnings.filterwarnings("ignore")

In [70]:
from ase.units import kB
import tqdm
import time

def run_metropolis(model, 
                    init, 
                    n_chains,
                    n_steps,
                    id_run,
                    energy_type,
                    temperature,
                    mixture,
                    nn_predictor=None,
                    frac_computed=0.2,
                    dim=12,
                    init_weights=None,
                    update_weights=True,
                    scheduler_weights=100,
                    alpha=0.5,
                    return_ratios=False,
                    return_proposals=True,
                    with_tqdm=True,
                    device='cpu',
                    folder_name=None,
                 ):

    assert init[:, :-2].shape[1] == dim
    assert init.shape[0] == n_chains

    if mixture:
        try:
            assert len(model.models) > 1
        except:
            raise RuntimeError("Model is not a mixture")
    
    print("Running Metropolis-Hastings")

    x_init = init[:, :dim]
    u_init = init[:, dim]
    isomer_init = init[:, dim+1]

    beta = 1 / (kB * temperature)

    print("Number of chains: {:d}".format(n_chains))
    print("Number of steps: {:d}".format(n_steps))
    print("Temperature: {:d}K".format(temperature))
    print("Energy Type: {:s}".format(energy_type))

    if "mlp" in energy_type:
        #TODO: Add calculator for MLP
        
        if (nn_predictor is None):
            raise RuntimeError("No model to calculate energy")
        
        if mixture:
            print('Mixture model')
            model_mlp_is0, model_mlp_is1 = nn_predictor
        else:
            if(isomer_init.sum()==0):
                model_mlp_is0 = nn_predictor
            else:
                model_mlp_is1 = nn_predictor
        
        print("Use Neural Predictor: True")
    
    else:
        
        print("No Neural Predictor")

    if ("dft" in energy_type) or ("emt" in energy_type):

        use_calc = True

        xs_calc = []
        us_calc = []
        isomers_calc = []
        inds_calc = []

        if "emt" in energy_type:

            from flonacomldft.dft_calculator import EMTCalculator
            from flonacomldft.internal_coordinates import Coordinates_mapping

            coord_mapping = Coordinates_mapping(etype='emt')
            calculator = EMTCalculator()

            print("Use EMT Calculator: {:s}".format(str(use_calc)))

        if "dft" in energy_type:

            # import gpaw.mpi as mpi
            # from flonacomldft.dft_calculator import DFTCalculator
            # from flonacomldft.internal_coordinates import Coordinates_mapping
        # 
            # coord_mapping = Coordinates_mapping(etype='dft')
            # calculator = DFTCalculator()
# 
            # #mpi.world.barrier()
# 
            # if folder_name is not None:
            #     calculator.initialize_calculator(foldername=folder_name)
            # else:
            #     calculator.initialize_calculator()

            print("Use DFT Calculator: {:s}".format(str(use_calc)))

    else:
    
        use_calc = False

    if return_ratios:
        ratios = []

    if return_proposals:
        xs_proposals = [init[:, :dim]]
        us_proposals = [init[:, dim]]
        isomers_proposals = [init[:, dim+1]]

    print("Mixture Model: {:s}".format(str(mixture)))

    if mixture:
        if init_weights is None:
            init_weights = [0.5, 0.5]
        weights = [init_weights]

    xs = []
    us = []
    accs = []
    isomers = []

    time_step_mcmc = []

    if with_tqdm == False:

        print("Step \t Acc Rate \t Population")

    if with_tqdm:
        pbar = tqdm.tqdm(range(n_steps))
    else:
        pbar = range(n_steps)

    def write_not_compute(x, isomer, id_run, dt, i):

        with open('not_computed_molecules.txt', 'w') as f:

            f.write("Molecule run {:s} step {:d} chain {:d} not computed\n".format(
                str(id_run),
                dt, 
                i))

    for dt in pbar:

        if mixture:
            x_new, isomer_new = model.sample(n_chains, return_mus=True)
        else:
            x_new = model.sample(n_chains)
            isomer_new = isomer_init.clone()

        x_new = x_new.clone().detach() #set float
        isomer_new = isomer_new.clone().detach() #set dype

        if return_proposals:
            xs_proposals.append(x_new)
            isomers_proposals.append(isomer_new)

        nll_x = model.nll(x_new)
        nll_x_init = model.nll(x_init)

        if "mlp" in energy_type:

            u_new = torch.zeros((n_chains, 1)).squeeze()

            if mixture:

                u_new[~isomer_new.bool()] = model_mlp_is0(x_new[~isomer_new.bool()]).reshape(1, -1)
                u_new[isomer_new.bool()] = model_mlp_is1(x_new[isomer_new.bool()]).reshape(1, -1)

            else:

                if(isomer_new.sum()==0):
                    u_new = model_mlp_is0(x_new)
                else:
                    u_new = model_mlp_is1(x_new)

            u_new = u_new.squeeze()

        if use_calc:

            ind_not_computed = torch.zeros(n_chains)
            ind_computed = torch.zeros(n_chains)

            if (energy_type == "dft") or (energy_type == "emt"):

                ind_computed = torch.ones(n_chains)
                u_new = torch.zeros((n_chains))

            else:

                n_computed = int(u_new.shape[0] * frac_computed)
                u_sort, ind_u_sort = u_new.sort()
                
                for idx in ind_u_sort[:n_computed]:
                    
                    ind_computed[idx] = 1

            for i, flag_computed in enumerate(ind_computed):

                if flag_computed:

                    try: 

                        molecule, logdetjac = coord_mapping.build_molecule_from_real_centered(
                            x_new[i].reshape(1, -1), 
                            isomer=isomer_new[i].int().item(),
                        )

                        input_calculator = {'atoms': molecule, }

                        if "dft" in energy_type:

                            input_calculator['filename'] = 'ag6_{:d}_{:d}_{:d}.out'.format(
                                id_run, dt, i
                            )

                            # mpi.world.barrier()

                        u = calculator.calculate_energy(**input_calculator)

                        u_new[i] = coord_mapping.compute_energy_in_new_frame(
                            u,
                            logdetjac*(-1),
                            temperature=temperature,
                        )

                        xs_calc.append(x_new[i])
                        us_calc.append(u_new[i])
                        isomers_calc.append(isomer_new[i])

                    except:

                        ind_not_computed[i] = 1
                        u_new[i] = 0.0

                        write_not_compute(x_new[i], isomer_new[i], id_run, dt, i)

            if use_calc and "dft" in energy_type:

                rank = mpi.world.rank

                if rank == 0:

                    time_step_mcmc.append(time.time())

                mpi.world.barrier()

            else:

                time_step_mcmc.append(time.time())

        if return_proposals:
            us_proposals.append(u_new)

        ratio = -beta * u_new + nll_x
        ratio += beta * u_init - nll_x_init
        ratio = torch.exp(ratio)

        if return_ratios:
            ratios.append( torch.min(ratio.clone().detach(), 
                            torch.ones_like(ratio)) )

        s = torch.rand_like(ratio)

        acc = s < torch.min(ratio.detach(), torch.ones_like(ratio))

        if use_calc:
    
            if ind_not_computed.sum() > 0:

                acc[ind_not_computed.bool()] = False

            inds_calc.append(ind_computed)

        x_new[~acc] = x_init[~acc]
        u_new[~acc] = u_init[~acc]

        if mixture:
        
            isomer_new[~acc] = isomer_init[~acc]
        
        else:

            isomer_new = isomer_init.clone()

        xs.append(x_new.clone().detach())
        us.append(u_new.clone().detach())
        accs.append(acc.clone().detach())
        isomers.append(isomer_new.clone().detach())

        x_init = x_new.clone().detach()
        u_init = u_new.clone().detach()
        isomer_init = isomer_new.clone().detach()

        if with_tqdm:
            pbar.set_description("Step {:d} \t Acceptance Rate {:.3f}".format(
                dt, acc.float().mean().item()))
        else:
            print("{:d} \t {:.3f} \t\t {:3f}".format( dt, 
                                        acc.float().mean().item(), 
                                        isomer_new.float().mean().item()
                                        )
            )

        if mixture and update_weights and dt > 0 and dt % scheduler_weights == 0:

            window_weights = torch.stack( [(~torch.stack(isomers).bool())[-scheduler_weights:].float().mean(),
                              torch.stack(isomers)[-scheduler_weights:].float().mean()] )

            new_weights = alpha * model.weights.clone() + (1 - alpha) * window_weights.clone()

            if torch.all(new_weights < 0.75):

                model.weights = new_weights.clone()

            weights.append(model.weights.clone().detach())

    time_mcmc = np.zeros(n_steps)

    if use_calc and "dft" in energy_type:

        rank = mpi.world.rank
        ranks = np.arange(mpi.world.size)
        comm = mpi.world.new_communicator(ranks)

        mpi.world.barrier()

        if rank == 0:

            time_mcmc = np.array(time_step_mcmc)

        comm.broadcast(time_mcmc, 0)

    to_return = {
        'xs': xs,
        'us': us,
        'accs': accs,
        'isomers': isomers,
        'time_mcmc': time_mcmc,
    }

    if return_ratios:
        to_return['ratios'] = ratios

    if return_proposals:
        to_return['xs_proposals'] = xs_proposals
        to_return['us_proposals'] = us_proposals
        to_return['isomers_proposals'] = isomers_proposals

    if use_calc:
            
        to_return['xs_calc'] = xs_calc
        to_return['us_calc'] = us_calc
        to_return['isomers_calc'] = isomers_calc
        to_return['inds_calc'] = inds_calc

    if mixture and update_weights:
        to_return['weights'] = weights

    return to_return

In [71]:
n_chains = 10
n_steps = 20
xs_init = flow_model.sample(n_chains)
us_init = mlp_model[0](xs_init)
isomers_init = torch.zeros(n_chains, 1)

init = torch.cat([xs_init, us_init, isomers_init], dim=1)

In [72]:
run_metropolis(model = flow_model, 
                init = init, 
                n_chains = n_chains,
                n_steps = n_steps,
                id_run = 'N/A', 
                energy_type = 'emt-mlp', 
                temperature = 350,
                mixture = True,
                nn_predictor = mlp_model,
                frac_computed = 0.2,
                dim = 12,
                init_weights = None,
                update_weights = True,
                scheduler_weights = 5,
                alpha = 0.5,
                return_ratios = False,
                return_proposals = True,
                with_tqdm = False,
                folder_name = None,
                )

Running Metropolis-Hastings
Number of chains: 10
Number of steps: 20
Temperature: 350K
Energy Type: emt-mlp
Mixture model
Use Neural Predictor: True
Use EMT Calculator: True
Mixture Model: True
Step 	 Acc Rate 	 Population
0 	 0.600 		 0.000000


1 	 0.100 		 0.000000
2 	 0.300 		 0.000000
3 	 0.200 		 0.000000
4 	 0.300 		 0.000000
5 	 0.200 		 0.000000
6 	 0.200 		 0.000000
7 	 0.100 		 0.000000
8 	 0.400 		 0.000000
9 	 0.200 		 0.000000
10 	 0.300 		 0.000000
11 	 0.200 		 0.000000
12 	 0.000 		 0.000000
13 	 0.200 		 0.000000
14 	 0.200 		 0.000000
15 	 0.100 		 0.000000
16 	 0.200 		 0.000000
17 	 0.000 		 0.000000
18 	 0.500 		 0.000000
19 	 0.300 		 0.000000


In [ ]:
# from flonacomldft.sampling import run_metropolis
# 
# run_metropolis(model = flow_model, 
#                 init = init, 
#                 n_chains = n_chains,
#                 n_steps = 2,
#                 id_run = 'N/A', 
#                 energy_type = 'emt-mlp', 
#                 mixture = False, 
#                 T = 350,
#                 mlp_models = [mlp_model],
#                 frac_dft = 0.2,
#                 dim = 12,
#                 #init_weights = None,
#                 return_ratio = False,
#                 return_proposals = True,
#                 with_tqdm = False,
#                 dft_folder_name = None,
#                 )

In [ ]:
torch.ones(5), torch.ones((5, 1)).squeeze(), torch.ones((5))

(tensor([1., 1., 1., 1., 1.]),
 tensor([1., 1., 1., 1., 1.]),
 tensor([1., 1., 1., 1., 1.]))

In [ ]:
torch.tensor(1.0).int().item()

1